# 16 — Modelos de Defasagens Distribuídas (DLM)

Segunda metodologia do TCC. O objetivo é estimar o perfil contemporâneo/defasado e o efeito acumulado no horizonte K. Esta etapa não refaz Granger e não interpreta associações como causalidade estrutural.

## 1. Protocolo e formulação
Partimos de $Y_{it}=\alpha_i+\lambda_t+\sum_{k=0}^K\beta_kX_{i,t-k}+\varepsilon_{it}$. As regras de estimabilidade, K, Almon, FE, inferência e FDR foram congeladas em `docs/dlm_protocolo.md` antes da interpretação.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from dlm import *
pd.set_option('display.max_columns',80)

## 2. Validação do painel
A pipeline exige 27 UFs, 144 meses no período total, ausência de duplicidade UF-mês e reconstrói a taxa como 100 × carteira inadimplente / carteira ativa.

In [ ]:
d=load_panel()
print(d.shape,d.uf.nunique(),d.data_base.min(),d.data_base.max())
assert len(d)==3888
assert not d.duplicated(['uf','data_base']).any()

## 3. Lags sem leakage
Todos os shifts são feitos dentro de UF. O primeiro mês de cada UF deve ter lag 1 ausente.

In [ ]:
cols=exposure_columns(d)
assert sanity_lags(d,cols[0])
print('Leakage check: OK; exposições:',len(cols))

## 4. Estimabilidade, DLM irrestrito e Almon
A análise cobre total, grupos e tipologias. K ∈ {3,6,9,12} é selecionado por BIC, não por significância. Almon é acionado por multicolinearidade/instabilidade dos lags; q ∈ {2,3}.

In [ ]:
A,S,C=run()
display(A)
display(S)
display(C.head(40))

## 5. Efeito acumulado, teste conjunto e FDR
A interpretação prioriza o teste conjunto e $\sum_{k=0}^K\beta_k$, com incerteza obtida da matriz completa de covariância. BH/FDR é aplicado por período × nível. P nominal e ajustado permanecem visíveis.

In [ ]:
if len(S):
    display(S[['periodo','nivel','exposicao','metodo','K','q','efeito_contemporaneo','efeito_acumulado','ci_acum_low','ci_acum_high','p_acumulado','q_p_acumulado','p_conjunto','q_p_conjunto']].sort_values(['periodo','nivel','q_p_acumulado']))

## 6. Diagnósticos, robustez e limites
A versão final deve classificar os modelos por autocorrelação, heterocedasticidade, dependência transversal, multicolinearidade e estabilidade; testar a robustez limitada definida no protocolo; e discutir a limitação de 27 clusters. Resultados não devem ser promovidos ou descartados com base em significância.

## 7. Relação com Granger e SARIMAX
DLM e Granger respondem perguntas distintas. Um lag nominal no DLM não contradiz Granger. Os horizontes e exposições mais estáveis nesta etapa podem orientar a especificação prévia do SARIMAX, que não é executado neste notebook.